<h1 style="text-align: center;">[Your Project Title]</h1>
<h3 style="text-align: center;">[Your Name]</h3>

---

## **Section 0. Setup**

### **0.1 Import Library**

In [23]:
# common library
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

# Feature Engineering
from sklearn.preprocessing import MinMaxScaler, RobustScaler
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer
from category_encoders import BinaryEncoder
from feature_engine.outliers import Winsorizer, OutlierTrimmer
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import SelectKBest
from sklearn.model_selection import train_test_split
from feature_engine.selection import DropFeatures
from feature_engine.datetime import DatetimeFeatures

# Model and pipelining
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

# Evaluation
from sklearn.metrics import ConfusionMatrixDisplay, classification_report
from sklearn.model_selection import cross_validate
from sklearn.model_selection import RandomizedSearchCV

# Imbalance
from imblearn.over_sampling import RandomOverSampler, SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline

### **0.2 Global Configuration**

In [2]:
RANDOM_STATE = 42
pd.set_option('display.max_columns', None)

## **Section 1. Business Understanding**

### **1.1 Context**
Kamar hotel itu seperti barang yang punya tanggal kedaluwarsa sangat cepat. Jika satu kamar tidak terjual malam ini, kesempatan untuk mendapatkan uang dari kamar tersebut di malam ini sudah hilang selamanya. 

Di Portugal, hotel-hotel menghadapi masalah besar karena banyak calon customer yang membatalkan pesanan mereka secara mendadak. Ketika hal ini terjadi, pihak hotel langsung terkena dua dampak buruk:
1. **Rugi Bandar (Kehilangan Pendapatan):** Kamar yang batal dipesan mendadak sering kali berakhir kosong karena hotel kehabisan waktu untuk mencari customer pengganti.
2. **Kacau di Operasional:** Manajemen hotel jadi kesulitan memprediksi kebutuhan harian—mulai dari jumlah bahan makanan yang harus disiapkan di restoran, hingga jumlah staf pembersih kamar yang perlu dijadwalkan masuk kerja.

Oleh karena itu, **Revenue Management Team** dan **Hotel General Manager** membutuhkan sistem yang bisa memprediksi apakah seorang customer akan membatalkan pesanannya atau tidak sebelum tanggal kedatangan.

### **1.2 Problem Statements**
Berdasarkan konteks di atas, masalah utama yang dihadapi oleh manajemen hotel dapat dirumuskan melalui pertanyaan-pertanyaan berikut:
1. Bagaimana cara mendeteksi customer yang memiliki probabilitas tinggi untuk membatalkan pesanan (`is_canceled` = 1) sebelum tanggal kedatangan mereka secara akurat?
2. Bagaimana cara mengoptimalkan alokasi kamar hotel agar tidak terjadi kekosongan akibat pembatalan, tanpa merusak reputasi hotel akibat *overbooking* yang berlebihan?


### **1.3 Goals**
Proyek ini memiliki target utama sebagai berikut:
1. **Membangun Model Prediktif:** Mengembangkan model machine learning klasifikasi yang mampu memprediksi dengan akurat apakah suatu pemesanan akan dibatalkan (`is_canceled` = 1) atau tidak (`is_canceled` = 0).
2. 

### **1.4 Analytical Approach**
Pendekatan analitis yang akan digunakan adalah **Supervised Machine Learning - Binary Classification**. 

Kita akan menganalisis data customer (seperti rekam jejak pembatalan sebelumnya, segmen pasar, tipe deposit, dan perubahan pesanan) untuk mengidentifikasi pola perilaku yang membedakan antara customer yang tetap datang (*check-in*) dengan customer yang melakukan pembatalan (*cancel*). Model Classification ini nantinya akan mengeluarkan probabilitas pembatalan untuk setiap pemesanan baru.

### **1.5 Metric Evaluation (Business Metric vs Machine Learning Metric)**
Dalam kasus binary classification ini, terdapat beberapa kesalahan yang mungkin dihasilkan oleh model:


### **1.6 Success Criteria**
Proyek machine learning ini akan dinyatakan sukses dan layak diimplementasikan jika memenuhi kriteria berikut:
* **Technical Aspect:** Model final mampu menghasilkan skor **F1-Score / Recall minimal 80%** pada data pengujian (*test set*).
* **Business Aspect:** Model mampu mengidentifikasi setidaknya 80% dari total pembatalan aktual, sehingga manajemen hotel dapat menyelamatkan potensi kerugian pendapatan kamar kosong secara signifikan dibandingkan tanpa menggunakan model.

---
## **Section 2. Data Understanding**
Pada bagian ini, kita melihat gambaran besar struktur data yang kita miliki untuk memahami ukuran dan format data secara sekilas sebelum melakukan proses pembersihan data.

### **2.1 General Information**

In [5]:
df = pd.read_csv('../data/raw/data_hotel_booking_demand.csv')
df.head()

,country,market_segment,previous_cancellations,booking_changes,deposit_type,days_in_waiting_list,customer_type,reserved_room_type,required_car_parking_spaces,total_of_special_requests,is_canceled
0,IRL,Offline TA/TO,0,0,No Deposit,0,Transient-Party,A,0,0,0
1,FRA,Online TA,0,0,No Deposit,0,Transient,A,0,2,0
2,PRT,Online TA,0,1,No Deposit,0,Transient,A,0,2,0
3,NLD,Online TA,0,0,No Deposit,0,Transient,A,0,1,1
4,PRT,Online TA,0,2,No Deposit,0,Transient,A,0,2,0


In [11]:
print(f"Dimensi Dataset: {df.shape}")

Dimensi Dataset: (83573, 11)


In [16]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 83573 entries, 0 to 83572
Data columns (total 11 columns):
 #   Column                       Non-Null Count  Dtype
---  ------                       --------------  -----
 0   country                      83222 non-null  str  
 1   market_segment               83573 non-null  str  
 2   previous_cancellations       83573 non-null  int64
 3   booking_changes              83573 non-null  int64
 4   deposit_type                 83573 non-null  str  
 5   days_in_waiting_list         83573 non-null  int64
 6   customer_type                83573 non-null  str  
 7   reserved_room_type           83573 non-null  str  
 8   required_car_parking_spaces  83573 non-null  int64
 9   total_of_special_requests    83573 non-null  int64
 10  is_canceled                  83573 non-null  int64
dtypes: int64(6), str(5)
memory usage: 9.7 MB


### **2.2 Feature Information**

| Feature | Description | Impact to Business |
| :--- | :--- | :--- |
| `is_canceled` *(Target)* | Indikasi apakah pemesanan dibatalkan atau tidak (1 = Batal, 0 = Check-in). | **Variabel Target.** Menentukan tingkat kerugian hunian kamar hotel. Model akan memprediksi nilai ini. |
| `country` | Negara asal customer yang melakukan pemesanan. | Membantu menganalisis perilaku pembatalan berdasarkan basis geografis. Customer internasional dari negara jauh umumnya memiliki risiko batal lebih rendah karena melibatkan tiket pesawat yang sulit dibatalkan. |
| `market_segment` | Saluran pemasaran tempat pemesanan dilakukan (misal: Online TA, Offline TA, Direct, Groups). | Segmen pasar menentukan komitmen customer. Pemesanan melalui agen travel online (Online TA) atau grup memiliki kecenderungan pembatalan berbeda dibanding pemesanan langsung (*Direct*). |
| `previous_cancellations` | Jumlah pemesanan yang pernah dibatalkan oleh customer tersebut sebelum pesanan saat ini. | Menunjukkan history jejak loyalitas dan perilaku customer. Customer yang sudah sering membatalkan pesanan sebelumnya memiliki risiko pembatalan berulang yang jauh lebih tinggi. |
| `booking_changes` | Jumlah perubahan yang dilakukan pada detail pemesanan sejak dibuat hingga check-in. | Mencerminkan tingkat keseriusan dan keterlibatan (*engagement*) customer. Customer yang aktif mengubah detail pesanan biasanya sangat berniat untuk datang menginap. |
| `deposit_type` | Jenis deposit yang diberikan untuk menjamin pemesanan (*No Deposit, Non Refund, Refundable*). | Mempengaruhi risiko finansial deposit_type jika batal. Customer yang memilih tipe *Non-Refundable* (uang hangus) kemungkinan besar tidak akan membatalkan pemesanannya. |
| `days_in_waiting_list` | Jumlah hari pemesanan berada di daftar tunggu sebelum akhirnya dikonfirmasi ke customer. | Semakin lama customer berada di daftar tunggu (*waiting list*), semakin besar kemungkinan mereka mencari alternatif hotel lain yang sehingga risiko batal meningkat. |
| `customer_type` | Kategori tipe pemesanan (*Transient, Transient-Party, Contract, Group*). | Membantu membedakan perilaku individu vs institusi/kontrak jangka panjang. Customer individu (*Transient*) biasanya lebih fleksibel bertindak tidak terduga dibanding kontrak korporat. |
| `reserved_room_type` | Kode jenis kamar yang dipesan oleh customer (disamarkan dengan kode huruf: A, B, C, dst). | Jenis kamar memengaruhi ekspektasi harga dan ketersediaan. Kamar mewah/tipe tertentu yang dibatalkan memiliki dampak finansial lebih besar dibanding kamar tipe standar. |
| `required_car_parking_spaces` | Jumlah tempat parkir mobil yang diminta oleh customer. | Merupakan indikator niat kedatangan yang kuat. Customer yang secara spesifik meminta ruang parkir biasanya akan datang karena perjalanan sudah direncanakan matang. |
| `total_of_special_requests` | Jumlah permintaan khusus yang diajukan customer (misal: ranjang twin, lantai tinggi). | Semakin banyak permintaan khusus yang diajukan customer, semakin tinggi keterlibatan emosional mereka dengan kunjungan tersebut, memperkecil peluang pembatalan. |

In [39]:
# Cek nilai unik dan distribusi feature kategorikal
categorical_features = ['country', 'market_segment', 'deposit_type', 'customer_type', 'reserved_room_type']

for i in categorical_features:
    print(f"\n--- Kolom: {i} ---")
    print(f"Total Kategori Unique: {df[i].nunique()}")
    print("Distribusi Semua Kategori:")

    counts = df[i].value_counts(dropna=False)
    
    for i, (val, count) in enumerate(zip(counts.index, counts.values)):
        print(f" {i+1}. {val}: {count}")



--- Kolom: country ---
Total Kategori Unique: 162
Distribusi Semua Kategori:
 1. PRT: 34097
 2. GBR: 8495
 3. FRA: 7307
 4. ESP: 5996
 5. DEU: 5116
 6. ITA: 2658
 7. IRL: 2340
 8. BEL: 1648
 9. BRA: 1553
 10. USA: 1472
 11. NLD: 1433
 12. CHE: 1201
 13. CN: 886
 14. AUT: 873
 15. SWE: 724
 16. CHN: 709
 17. POL: 638
 18. ISR: 463
 19. RUS: 435
 20. NOR: 431
 21. nan: 351
 22. ROU: 341
 23. FIN: 316
 24. DNK: 308
 25. AUS: 301
 26. AGO: 243
 27. LUX: 181
 28. MAR: 179
 29. TUR: 163
 30. ARG: 154
 31. HUN: 138
 32. JPN: 129
 33. CZE: 117
 34. IND: 104
 35. KOR: 102
 36. GRC: 94
 37. SRB: 81
 38. DZA: 80
 39. HRV: 73
 40. IRN: 63
 41. ZAF: 60
 42. LTU: 59
 43. MEX: 57
 44. EST: 55
 45. BGR: 55
 46. NZL: 49
 47. COL: 47
 48. CHL: 43
 49. MOZ: 42
 50. UKR: 42
 51. SVN: 42
 52. ISL: 41
 53. SVK: 41
 54. THA: 40
 55. ARE: 38
 56. SAU: 37
 57. LVA: 36
 58. TWN: 34
 59. CYP: 33
 60. TUN: 31
 61. SGP: 27
 62. PHL: 27
 63. IDN: 27
 64. HKG: 26
 65. LBN: 24
 66. URY: 23
 67. NGA: 22
 68. EGY: 21


Pada country terlihat terdapat NaN (nomor 21) sebanyak 351 

In [41]:
# Cek nilai unik dan sebaran feature numerikal
numerical_features = ['previous_cancellations', 'booking_changes', 'days_in_waiting_list', 'required_car_parking_spaces', 'total_of_special_requests']

for col in numerical_features:
    print(f"\n--- Kolom: {col} ---")
    print(f"Total Nilai Unique: {df[col].nunique(dropna=False)}")
    print("Distribusi Semua Nilai:")
    
    counts = df[col].value_counts(dropna=False)
    
    for i, (val, count) in enumerate(zip(counts.index, counts.values)):
        print(f" {i+1}. Nilai {val}: {count} ")



--- Kolom: previous_cancellations ---
Total Nilai Unique: 15
Distribusi Semua Nilai:
 1. Nilai 0: 79060 
 2. Nilai 1: 4207 
 3. Nilai 2: 86 
 4. Nilai 3: 46 
 5. Nilai 24: 33 
 6. Nilai 11: 28 
 7. Nilai 6: 19 
 8. Nilai 4: 19 
 9. Nilai 26: 18 
 10. Nilai 25: 17 
 11. Nilai 19: 12 
 12. Nilai 13: 10 
 13. Nilai 14: 10 
 14. Nilai 5: 7 
 15. Nilai 21: 1 

--- Kolom: booking_changes ---
Total Nilai Unique: 19
Distribusi Semua Nilai:
 1. Nilai 0: 70873 
 2. Nilai 1: 8963 
 3. Nilai 2: 2652 
 4. Nilai 3: 639 
 5. Nilai 4: 260 
 6. Nilai 5: 90 
 7. Nilai 6: 39 
 8. Nilai 7: 23 
 9. Nilai 8: 10 
 10. Nilai 10: 5 
 11. Nilai 9: 4 
 12. Nilai 13: 4 
 13. Nilai 17: 2 
 14. Nilai 12: 2 
 15. Nilai 14: 2 
 16. Nilai 16: 2 
 17. Nilai 21: 1 
 18. Nilai 20: 1 
 19. Nilai 15: 1 

--- Kolom: days_in_waiting_list ---
Total Nilai Unique: 115
Distribusi Semua Nilai:
 1. Nilai 0: 80988 
 2. Nilai 39: 166 
 3. Nilai 58: 104 
 4. Nilai 31: 93 
 5. Nilai 44: 93 
 6. Nilai 46: 66 
 7. Nilai 35: 66 
 8. Nil

In [46]:
# Cek proporsi kelas target
target_counts = df['is_canceled'].value_counts()
target_percentage = df['is_canceled'].value_counts(normalize=True) * 100

for val, count, pct in zip(target_counts.index, target_counts.values, target_percentage.values):
    status = "Cancel" if val == 1 else "Check-in"
    print(f"[Class {val}] {status}: {count} data ({pct:.2f}%)")

[Class 0] Check-in: 52795 data (63.17%)
[Class 1] Cancel: 30778 data (36.83%)


### **2.3 Statistics Summary**

#### 2.3.1 Statistik Deskriptif Fitur Numerik

In [42]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
previous_cancellations,83573.0,0.086798,0.841011,0.0,0.0,0.0,0.0,26.0
booking_changes,83573.0,0.220897,0.648635,0.0,0.0,0.0,0.0,21.0
days_in_waiting_list,83573.0,2.330561,17.673051,0.0,0.0,0.0,0.0,391.0
required_car_parking_spaces,83573.0,0.062999,0.246919,0.0,0.0,0.0,0.0,8.0
total_of_special_requests,83573.0,0.573211,0.795163,0.0,0.0,0.0,1.0,5.0
is_canceled,83573.0,0.368277,0.482340,0.0,0.0,0.0,1.0,1.0


#### 2.3.2 Statistik Deskriptif Fitur Kategorikal

In [43]:
df.describe(include='object').T

/var/folders/f7/c5qn45190zdds3jh0t1vyfmh0000gn/T/ipykernel_3482/1760094569.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include='object').T


,count,unique,top,freq
country,83222,162,PRT,34097
market_segment,83573,8,Online TA,39460
deposit_type,83573,3,No Deposit,73352
customer_type,83573,4,Transient,62732
reserved_room_type,83573,10,A,60041


## **Section 3. Data Cleaning**

### **3.1 Missing Values**
Mengidentifikasi kolom yang punya data hilang dan menentukan strategi menanganinya (drop, imputasi, atau dibiarkan dengan alasan tertentu).

In [48]:
df.isnull().sum()

country                        351
market_segment                   0
previous_cancellations           0
booking_changes                  0
deposit_type                     0
days_in_waiting_list             0
customer_type                    0
reserved_room_type               0
required_car_parking_spaces      0
total_of_special_requests        0
is_canceled                      0
dtype: int64

In [ ]:
# drop atau impute valuenya jadi 'unknown'

### **3.2 Duplicated Values**

In [54]:
total_duplicates = df.duplicated().sum()
percentage_duplicates = (total_duplicates / len(df)) * 100

print(total_duplicates)
print(f"Percentage: {percentage_duplicates:.2f}%")

73371
Percentage: 87.79%


Sebenarnya tidak ada duplikat karena tidak adanya ID Transaksi unik

### **3.3 Data Consistency Check**
- Spelling errors / typo pada kategori
- Inkonsistensi kapitalisasi & format penulisan
- Whitespace tersembunyi

In [56]:
# ilangin whitespace tersembunyi di awal/akhir string (kalau ada) dan menyeragamkan case
categorical_cols = df.select_dtypes(include='object').columns
for col in categorical_cols:
    df[col] = df[col].astype(str).str.strip()

print(f"Jumlah baris sebelum filter 'Undefined': {len(df)}")

df = df[df['market_segment'] != 'Undefined']
print(f"Jumlah baris setelah filter 'Undefined': {len(df)}")

Jumlah baris sebelum filter 'Undefined': 83572
Jumlah baris setelah filter 'Undefined': 83572


/var/folders/f7/c5qn45190zdds3jh0t1vyfmh0000gn/T/ipykernel_3482/3623355706.py:2: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = df.select_dtypes(include='object').columns


### **3.4 Identify Anomaly Values**
- Check Distribution (Numerical Variable)
- Check Cardinality (Categorical Variable)

## **Section 4. Exploratory Data Analysis (EDA)**

> 🎯 *Tujuan:* Menggali pola dan hubungan dalam data training untuk membangun intuisi sebelum masuk ke tahap modeling.

**5.1 Univariate Analysis**
- Distribusi target
- Distribusi fitur numerik
- Distribusi fitur kategorikal

> 🎯 *Tujuan:* Memahami karakteristik tiap variabel secara individual, termasuk seberapa seimbang distribusi target.

**5.2 Bivariate Analysis (terhadap Target)**

> 🎯 *Tujuan:* Mencari pola hubungan antara tiap fitur dengan target, untuk menjawab langsung Problem Statement di Section 1.2.

> 📌 Ini bagian paling penting untuk menjawab Problem Statement di Section 1.2 — cari pola antara tiap fitur dengan target, bukan sekadar plot tanpa insight.

**5.3 Correlation & Multicollinearity Check**

> 🎯 *Tujuan:* Mengecek hubungan antar fitur untuk mendeteksi multikolinearitas yang bisa mengganggu interpretasi model nanti.

**5.4 Multivariate / Interaction Analysis (opsional)**

> 🎯 *Tujuan:* Menelusuri interaksi antara beberapa fitur sekaligus untuk pola yang lebih kompleks dari yang bisa ditangkap analisis dua arah.

## **Section 5. Data Preparation**

> 🎯 *Tujuan:* Mengubah data mentah menjadi bentuk siap pakai untuk pemodelan (numerik, terskala, tanpa kategori yang belum di-encode).

**5.1 Initialization**
- Initialization function
- Define Feature and Target

> 🎯 *Tujuan:* Menyiapkan fungsi bantu dan mendefinisikan mana kolom fitur (X) dan target (y) sebelum transformasi dimulai.

**5.2 Constructing `Training` and `Testing` Data (from `Seen` Dataset)**

> 🎯 *Tujuan:* Membagi data `Seen` menjadi training dan testing untuk keperluan pengembangan dan evaluasi model.

**5.3 Handling Imbalanced Data (jika relevan)**

> 🎯 *Tujuan:* Menangani ketimpangan proporsi kelas target supaya model tidak bias ke kelas mayoritas.

> 📌 Cek proporsi kelas target di Section 5.1. Kalau timpang (misal 90:10), pertimbangkan strategi seperti class_weight, SMOTE, atau undersampling — **tapi ingat, teknik resampling hanya boleh diterapkan pada data training**, tidak pernah pada data testing/unseen, supaya evaluasi tetap realistis.

**5.4 Data Transformation (Feature Engineering)**

> 🎯 *Tujuan:* Melakukan encoding, scaling, atau transformasi lain agar data sesuai kebutuhan algoritma yang dipakai.

**5.5 Feature Selection**

> 🎯 *Tujuan:* Memilih fitur yang paling relevan/berkontribusi untuk mengurangi noise dan risiko overfitting.

**5.6 Overview**

> 🎯 *Tujuan:* Merangkum hasil akhir data preparation (bentuk data final) sebelum masuk ke tahap Model Development.

## **Section 6. Model Development**

> 🎯 *Tujuan:* Membangun, membandingkan, dan menyempurnakan model machine learning menggunakan data.

**6.1 Initialization**
- Initialization Function
- Create Custom Metrics
- Define Cross-Validation Strategy
- Create a workflow of the experiment

> 🎯 *Tujuan:* Menyiapkan fungsi metrik custom dan strategi cross-validation yang dipakai konsisten di seluruh eksperimen model.

> 📌 Tentukan strategi CV secara eksplisit (misal `StratifiedKFold` untuk klasifikasi dengan target tidak seimbang) dan simpan `RANDOM_STATE` yang sama dari Section 0. Ingat prinsip **CV-first**: bandingkan model lewat cross-validation dulu, baru evaluasi akhir di data testing — jangan sebaliknya.

**6.2 Developing the Model Pipeline**

> 🎯 *Tujuan:* Merangkai seluruh langkah preprocessing dan model ke dalam satu objek Pipeline yang konsisten dipakai ulang.

> 📌 Gunakan `Pipeline`/`ColumnTransformer` dari scikit-learn supaya seluruh langkah preprocessing (imputasi, encoding, scaling) ikut ter-*fit* hanya pada data training di setiap fold — ini mencegah data leakage antara fold CV.

**6.3 Model Benchmarking (Comparing model base performance)**

> 🎯 *Tujuan:* Membandingkan performa dasar beberapa algoritma (tanpa tuning) untuk memilih kandidat terbaik yang layak dituning lebih lanjut.

**6.4 Tune Model**

> 🎯 *Tujuan:* Mengoptimalkan hyperparameter dari model kandidat terbaik hasil benchmarking untuk meningkatkan performa.

**6.5 Analyze Model**
- Evaluate model on data testing
- Confusion Matrix / Threshold Analysis (Classification) atau Residual Analysis (Regression)
- Learning Curve Inspection

> 🎯 *Tujuan:* Mengevaluasi performa model secara mendalam di luar satu angka metrik utama, termasuk mengecek tanda overfitting/underfitting.

**6.6 Model Calibration (Classification Only)**

> 🎯 *Tujuan:* Menyesuaikan output probabilitas model supaya lebih merepresentasikan kemungkinan sebenarnya, penting saat threshold dipakai untuk keputusan bisnis.

**6.7 Model Explanation and Interpretation**
- Feature Importance (Tree Based Model) / Coefficient Regression (Regression Based Model)
- SHAP Value identification
- Counterfactual Analysis

> 🎯 *Tujuan:* Menjelaskan bagaimana model mengambil keputusan — penting untuk membangun kepercayaan stakeholder bisnis terhadap model.

## **Section 7. Model Deployment**

> 🎯 *Tujuan:* Menyiapkan model terlatih agar bisa dipakai di luar notebook, lengkap dengan dokumentasi teknis yang diperlukan.

**7.1 Export Model (joblib/pickle)**

> 🎯 *Tujuan:* Menyimpan pipeline terlatih ke dalam file yang bisa dimuat ulang tanpa perlu melatih ulang dari awal.

> 📌 Minimal, export pipeline lengkap (bukan cuma model) dengan `joblib.dump()` supaya preprocessing dan model tetap satu paket saat dipakai ulang.

**7.2 Deployment Checklist**
- Versi library yang digunakan
- Format input yang diharapkan model
- Cara memuat ulang pipeline

> 🎯 *Tujuan:* Mendokumentasikan hal teknis yang perlu diperhatikan tim lain saat model dipakai di lingkungan produksi.

## **Section 8. Model Implementation**

> 🎯 *Tujuan:* Menjelaskan cara pakai model di dunia nyata, batasannya, dan dampak bisnisnya lewat simulasi.

**8.1 How to implement the model?**

> 🎯 *Tujuan:* Menjelaskan langkah teknis memakai model untuk melakukan prediksi pada data baru.

**8.2 What are the limitations of the model?**

> 🎯 *Tujuan:* Mengakui batasan model secara jujur, termasuk skenario di mana prediksinya kurang bisa diandalkan.

**8.3 Business Calculation (Simulation using unseen data)**

> 🎯 *Tujuan:* Mensimulasikan dampak bisnis dari penggunaan model, memakai data `unseen` yang belum pernah dilihat selama proses modeling.

> 📌 Ini saatnya `unseen` data dipakai. Kaitkan hasil simulasi dengan metrik bisnis yang kamu tetapkan di Section 1.5 — misalnya, hitung estimasi kerugian akibat False Negative vs biaya operasional akibat False Positive, sesuai threshold yang dipilih.

## **Section 9. Conclusion and Recommendation**

> 🎯 *Tujuan:* Merangkum keseluruhan proyek dan menerjemahkan hasil teknis kembali ke bahasa yang dipahami stakeholder bisnis.

**9.1 Conclusion**
- Conclusion (Model)
- Conclusion (Business)

> 🎯 *Tujuan:* Merangkum temuan utama dari sisi performa model dan sisi dampak bisnis, menjawab kembali Goals di Section 1.3.

**9.2 Recommendation**
- Recommendation (Model)
- Recommendation (Business)

> 🎯 *Tujuan:* Memberikan rekomendasi tindak lanjut konkret berdasarkan temuan proyek, baik dari sisi teknis maupun bisnis.